In [20]:
# Parte 1a: Consolidar libertad_prensa_01 y libertad_prensa_02 con concat
# Parte 0: Importar librerías necesarias
import numpy as np
import pandas as pd
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
]

lista_df = [pd.read_csv(url) for url in archivos_anio]

# Normalizamos nombres de columnas a minúscula en cada DataFrame
for d in lista_df:
    d.columns = d.columns.str.lower()

df_anio = pd.concat(lista_df, ignore_index=True)
print("df_anio shape:", df_anio.shape)
df_anio.head()

df_anio shape: (3060, 4)


,codigo_iso,anio,indice,ranking
0,AFG,2001,35.5,59.0
1,AGO,2001,30.2,50.0
2,ALB,2001,NaN,NaN
3,AND,2001,NaN,NaN
4,ARE,2001,NaN,NaN


In [21]:
# Parte 1b: Detectar y limpiar el codigo_iso con dos nombres de país distintos
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')

# Detectamos automáticamente qué codigo_iso tiene más de un nombre de país asociado
conteo_por_codigo = df_codigos.groupby('codigo_iso')['pais'].nunique()
codigos_conflictivos = conteo_por_codigo[conteo_por_codigo > 1]
print("Código(s) ISO con más de un nombre de país:")
print(codigos_conflictivos)
print(df_codigos[df_codigos['codigo_iso'].isin(codigos_conflictivos.index)])

# El código ZWE aparece con "Zimbabue" (correcto) y "malo" (valor inválido)
# Eliminamos el registro inválido
df_codigos = df_codigos[df_codigos['pais'] != 'malo'].reset_index(drop=True)

print("\ndf_codigos limpio, shape:", df_codigos.shape)

Código(s) ISO con más de un nombre de país:
codigo_iso
ZWE    2
Name: pais, dtype: int64
    codigo_iso      pais
179        ZWE  Zimbabue
180        ZWE      malo

df_codigos limpio, shape: (180, 2)


In [22]:
# Parte 1c: Combinar df_anio con df_codigos usando merge (inner join por codigo_iso)
df = pd.merge(df_anio, df_codigos, on='codigo_iso', how='inner')

print("df final shape:", df.shape)
df.head()

df final shape: (3060, 5)


,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos


In [23]:
# Parte 2: Exploración inicial del DataFrame consolidado

# --- Estructura ---
print("Filas y columnas:", df.shape)
print("\nNombres de columnas:", df.columns.tolist())
print("\nTipos de datos:")
print(df.dtypes)
print("\nInfo general:")
df.info()

# --- Resumen estadístico ---
print("\nResumen estadístico:")
print(df.describe())

# País con el índice mínimo y máximo
fila_min_indice = df.loc[df['indice'].idxmin()]
fila_max_indice = df.loc[df['indice'].idxmax()]
print("\nMenor indice registrado:\n", fila_min_indice)
print("\nMayor indice registrado:\n", fila_max_indice)

# --- Datos faltantes ---
nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({'nulos': nulos, 'porcentaje (%)': porcentaje_nulos})
print("\nValores nulos por columna:")
print(resumen_nulos)

columnas_30pct_nulos = resumen_nulos[resumen_nulos['porcentaje (%)'] > 30]
print("\nColumnas con más del 30% de nulos:")
print(columnas_30pct_nulos if not columnas_30pct_nulos.empty else "Ninguna")

# --- Unicidad y duplicados ---
print("\nPaíses distintos:", df['pais'].nunique())
print("Años distintos:", df['anio'].nunique())
print("Filas duplicadas exactas:", df.duplicated().sum())

# --- Validación cruzada país <-> código ---
inconsistencias = df.groupby('codigo_iso')['pais'].nunique()
inconsistencias = inconsistencias[inconsistencias > 1]
print("\nCódigos ISO asociados a más de un país (tras la limpieza):")
print(inconsistencias if not inconsistencias.empty else "Ninguna — la limpieza de la Parte 1 funcionó correctamente")

Filas y columnas: (3060, 5)

Nombres de columnas: ['codigo_iso', 'anio', 'indice', 'ranking', 'pais']

Tipos de datos:
codigo_iso     object
anio            int64
indice        float64
ranking       float64
pais           object
dtype: object

Info general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   codigo_iso  3060 non-null   object 
 1   anio        3060 non-null   int64  
 2   indice      2664 non-null   float64
 3   ranking     2837 non-null   float64
 4   pais        3060 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 119.7+ KB

Resumen estadístico:
              anio        indice        ranking
count  3060.000000   2664.000000    2837.000000
mean   2009.941176    205.782316     477.930913
std       5.786024   2695.525264    6474.935347
min    2001.000000      0.000000       1.000000
25%    2005.000000     

In [24]:
# Parte 3a: Extremos de indice por año en Latinoamérica, usando un ciclo for
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']

df_america = df.loc[df['codigo_iso'].isin(america)]

resultados_for = []
for anio in sorted(df_america['anio'].dropna().unique()):
    datos_anio = df_america.loc[df_america['anio'] == anio]
    if datos_anio['indice'].notna().any():
        fila_min = datos_anio.loc[datos_anio['indice'].idxmin()]
        fila_max = datos_anio.loc[datos_anio['indice'].idxmax()]
        resultados_for.append({
            'anio': anio,
            'pais_mayor_libertad': fila_min['pais'],
            'indice_min': fila_min['indice'],
            'pais_menor_libertad': fila_max['pais'],
            'indice_max': fila_max['indice']
        })

df_resultado_for = pd.DataFrame(resultados_for)
df_resultado_for.head(10)

,anio,pais_mayor_libertad,indice_min,pais_menor_libertad,indice_max
0,2001,Canadá,0.80,Cuba,90.30
1,2002,Trinidad y Tobago,1.00,Cuba,97.83
2,2003,Trinidad y Tobago,2.00,Argentina,35826.00
3,2004,Trinidad y Tobago,2.00,Cuba,87.00
4,2005,Bolivia,4.50,Cuba,95.00
5,2006,Canadá,4.88,Cuba,96.17
6,2007,Canadá,3.33,Cuba,88.33
7,2008,Canadá,3.70,Cuba,94.00
8,2009,Estados Unidos,6.75,Cuba,78.00
9,2012,Jamaica,9.88,Cuba,71.64


In [25]:
# Parte 3b: Mismo resultado, usando groupby (enfoque vectorizado, sin for)
idx_min = df_america.groupby('anio')['indice'].idxmin().dropna()
idx_max = df_america.groupby('anio')['indice'].idxmax().dropna()

paises_min = df_america.loc[idx_min, ['anio', 'pais', 'indice']].rename(
    columns={'pais': 'pais_mayor_libertad', 'indice': 'indice_min'})
paises_max = df_america.loc[idx_max, ['anio', 'pais', 'indice']].rename(
    columns={'pais': 'pais_menor_libertad', 'indice': 'indice_max'})

df_resultado_groupby = pd.merge(paises_min, paises_max, on='anio').sort_values('anio')
df_resultado_groupby.head(10)

,anio,pais_mayor_libertad,indice_min,pais_menor_libertad,indice_max
0,2001,Canadá,0.80,Cuba,90.30
1,2002,Trinidad y Tobago,1.00,Cuba,97.83
2,2003,Trinidad y Tobago,2.00,Argentina,35826.00
3,2004,Trinidad y Tobago,2.00,Cuba,87.00
4,2005,Bolivia,4.50,Cuba,95.00
5,2006,Canadá,4.88,Cuba,96.17
6,2007,Canadá,3.33,Cuba,88.33
7,2008,Canadá,3.70,Cuba,94.00
8,2009,Estados Unidos,6.75,Cuba,78.00
9,2012,Jamaica,9.88,Cuba,71.64


In [26]:
# Verificación: ambos enfoques deben dar el mismo resultado
comparacion = (
    df_resultado_for.set_index('anio')[['pais_mayor_libertad', 'pais_menor_libertad']]
    .equals(df_resultado_groupby.set_index('anio')[['pais_mayor_libertad', 'pais_menor_libertad']])
)
print("¿El resultado del for y de groupby coinciden?", comparacion)

¿El resultado del for y de groupby coinciden? True


In [27]:
# Parte 4: pivot_table de indice máximo por país y año
pivot_indice = pd.pivot_table(
    df,
    index='pais',
    columns='anio',
    values='indice',
    aggfunc='max',
    fill_value=0
)

print("Forma de la tabla pivot:", pivot_indice.shape)
pivot_indice.head(10)

Forma de la tabla pivot: (179, 16)


anio,2001,2002,2003,2004,2005,2006,2007,2008,2009,2012,2013,2014,2015,2017,2018,2019
pais,,,,,,,,,,,,,,,,
Afghanistán,35.5,40.17,28.25,39.17,44.25,56.50,59.25,54.25,51.67,37.36,37.07,37.44,37.75,39.46,37.28,36.55
Albania,0.0,6.50,11.50,14.17,18.00,25.50,16.00,21.75,21.50,30.88,29.92,28.77,29.92,29.92,29.49,29.84
Alemania,1.5,1.33,2.00,4.00,5.50,5.75,4.50,3.50,4.25,10.24,10.23,11.47,14.80,14.97,14.39,14.60
Algeria,31.0,33.00,43.50,40.33,40.00,40.50,31.33,49.56,47.33,36.54,36.26,36.63,41.69,42.83,43.13,45.75
Andorra,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,6.82,6.82,19.87,19.87,21.03,22.21,24.63
Angola,30.2,28.00,26.50,18.00,21.50,26.50,29.50,36.50,28.50,37.80,36.50,37.84,39.89,40.42,38.35,34.96
Antigua y Barbuda,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,20.81,0.00,0.00,0.00,0.00,0.00
Arabia Saudita,62.5,71.50,79.17,66.00,76.00,59.75,61.75,76.50,61.50,56.88,58.30,59.41,59.72,66.02,63.13,65.88
Argentina,12.0,15.17,35826.00,13.67,17.30,24.83,14.08,11.33,16.35,25.67,25.27,26.11,25.09,25.07,26.05,28.30


In [28]:
# Parte 4a: País con mayor y menor indice global (excluyendo ceros del relleno)
maximo_por_pais = pivot_indice.max(axis=1)
minimo_por_pais_no_cero = pivot_indice.replace(0, np.nan).min(axis=1)

pais_mayor_indice = maximo_por_pais.idxmax()
pais_menor_indice = minimo_por_pais_no_cero.idxmin()

print(f"País con mayor indice en toda la tabla: {pais_mayor_indice} ({maximo_por_pais.max()})")
print(f"País con menor indice (distinto de 0): {pais_menor_indice} ({minimo_por_pais_no_cero.min()})")

País con mayor indice en toda la tabla: Kosovo (64536.0)
País con menor indice (distinto de 0): Austria (0.5)


In [29]:
# Parte 4b: Años con promedio de indice más alto y más bajo
promedio_por_anio = pivot_indice.mean(axis=0)

print(f"Año con promedio de indice más alto: {promedio_por_anio.idxmax()} ({promedio_por_anio.max():.2f})")
print(f"Año con promedio de indice más bajo: {promedio_por_anio.idxmin()} ({promedio_por_anio.min():.2f})")

Año con promedio de indice más alto: 2013 (449.11)
Año con promedio de indice más bajo: 2001 (20.03)


In [30]:
# Parte 4c: País con mayor variabilidad (max - min a través de los años)
variabilidad = pivot_indice.max(axis=1) - pivot_indice.min(axis=1)
pais_mas_variable = variabilidad.idxmax()

print(f"País con mayor variabilidad: {pais_mas_variable} (diferencia: {variabilidad.max()})")

País con mayor variabilidad: Kosovo (diferencia: 64536.0)


In [31]:
# Parte 4d: Países con indice constante a lo largo de los años (con al menos 2 años de dato real)
valores_no_cero = pivot_indice.replace(0, np.nan)
anios_con_dato = (valores_no_cero.notna()).sum(axis=1)
es_constante = valores_no_cero.nunique(axis=1) == 1

paises_constantes = pivot_indice.loc[es_constante & (anios_con_dato > 1)]
print("Países con indice constante en todos sus años registrados:")
print(paises_constantes.index.tolist() if len(paises_constantes) > 0 else "Ninguno")

Países con indice constante en todos sus años registrados:
Ninguno


In [32]:
# Parte 4e: Países sin ningún dato (todas las columnas en 0 tras el fill_value)
paises_sin_datos = pivot_indice[(pivot_indice == 0).all(axis=1)]
print("Países sin ningún dato registrado:")
print(paises_sin_datos.index.tolist() if len(paises_sin_datos) > 0 else "Ninguno")

Países sin ningún dato registrado:
Ninguno
